Downloads

In [1]:
! pip install nltk

     ---------------------------------------- 0.0/41.5 kB ? eta -:--:--
     --------- ------------------------------ 10.2/41.5 kB ? eta -:--:--
     -------------------------------------  41.0/41.5 kB 495.5 kB/s eta 0:00:01
     -------------------------------------- 41.5/41.5 kB 333.1 kB/s eta 0:00:00
     ---------------------------------------- 0.0/57.7 kB ? eta -:--:--
     ----------------------------------- ---- 51.2/57.7 kB 2.6 MB/s eta 0:00:01
     ---------------------------------------- 57.7/57.7 kB 1.0 MB/s eta 0:00:00
   ---------------------------------------- 0.0/1.5 MB ? eta -:--:--
   -- ------------------------------------- 0.1/1.5 MB 2.6 MB/s eta 0:00:01
   ----- ---------------------------------- 0.2/1.5 MB 3.5 MB/s eta 0:00:01
   ------------- -------------------------- 0.5/1.5 MB 4.7 MB/s eta 0:00:01
   ------------- -------------------------- 0.5/1.5 MB 4.7 MB/s eta 0:00:01
   ------------------- -------------------- 0.7/1.5 MB 4.2 MB/s eta 0:00:01
   -----------


[notice] A new release of pip is available: 24.0 -> 25.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Imports

In [ ]:
import nltk
from nltk.corpus import wordnet as wn

nltk.download('wordnet')

[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\LEGION\AppData\Roaming\nltk_data...


True

Document and Query

In [12]:
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
import string

nltk.download('punkt')
nltk.download('stopwords')
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\LEGION\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\LEGION\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\LEGION\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping tokenizers\punkt_tab.zip.


True

In [13]:
documents = [
    "The car was parked beside the bank of the river.",
    "She deposited money in the bank and then drove her car.",
    "The river flooded the fields after the heavy rain.",
    "He bought a new automobile to replace his old car.",
    "The financial institution offers loans to small businesses."
]
query = "car near bank"

In [14]:
stop_words = set(stopwords.words('english'))

# Tokenize and lowercase everything
tokens = set()
for doc in documents + [query]:
    words = word_tokenize(doc.lower())
    words = [w for w in words if w.isalnum() and w not in stop_words]
    tokens.update(words)

vocab = sorted(tokens)
print(vocab)

['automobile', 'bank', 'beside', 'bought', 'businesses', 'car', 'deposited', 'drove', 'fields', 'financial', 'flooded', 'heavy', 'institution', 'loans', 'money', 'near', 'new', 'offers', 'old', 'parked', 'rain', 'replace', 'river', 'small']


** 1) Query Expansion (Wordnet) **

In [15]:
def query_expansion(query):
    words = nltk.word_tokenize(query.lower())
    expanded_query = set(words)
    for word in words:
        for syn in wn.synsets(word):
            for lemma in syn.lemmas():
                expanded_query.add(lemma.name().replace('_',' '))
    return list(expanded_query)

print("Result of query expansion:")
print(query_expansion(query))

Result of query expansion:
['nearly', 'go up', 'trust', 'swear', 'motorcar', 'banking company', 'dear', 'elevator car', 'auto', 'approximate', 'machine', 'bank building', 'nigh', 'come near', 'depository financial institution', 'money box', 'near', 'about', 'automobile', 'cable car', 'well-nigh', 'come on', 'close', 'cant', 'railroad car', 'camber', 'banking concern', 'rely', 'deposit', 'railcar', 'draw close', 'coin bank', 'draw near', 'good', 'virtually', 'savings bank', 'gondola', 'penny-pinching', 'railway car', 'approach', 'skinny', 'cheeseparing', 'almost', 'car', 'bank', 'most']


** 2) Spelling Correction **

A. Edit Distance (Levenshtein Distance)

This method corrects misspelled words by computing the minimum number of single-character edits (insertions, deletions, or substitutions) required to transform one word into another. The closest word from a vocabulary (smallest distance) is selected as the correction.

In [22]:
def edit_distance_calculator(w1,w2):
    matrix = [[[0] for _ in range(len(w2)+1)] for _ in range(len(w1)+1)]
    for i in range(len(w1)+1):
        for j in range(len(w2)+1):
            if i==0:
                matrix[i][j]=j
            elif j==0:
                matrix[i][j]=i
            elif w1[i-1] == w2[j-1]:
                matrix[i][j]=matrix[i-1][j-1]
            else:
                matrix[i][j] = 1+min(matrix[i-1][j-1],matrix[i-1][j],matrix[i][j-1])
    return matrix[-1][-1]

def correct_word(word,vocab):
    correction = word
    min_distance = float('inf')
    for correct in vocab:
        distance = edit_distance_calculator(word,correct)
        if distance < min_distance:
            min_distance = distance
            correction = correct
    return correction

print('Edit Distance Correction:')
print(correct_word('bunk',vocab))



Edit Distance Correction:
bank


B. K-Gram Index A k-gram is a substring of length k used to index and compare words. The K-gram index maps these substrings to words. If a user enters a misspelled word, we generate its k-grams and look for vocabulary words sharing many of these, assuming higher overlap implies greater similarity.